# Debugging

In [ ]:
import pandas as pd
import geopandas as gpd
import shapely
import numpy as np
import json
import os
from functools import partial
from tqdm import tqdm

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.ticker import MaxNLocator

import nomad.io.base as loader
import nomad.stop_detection.utils as utils
from nomad.stop_detection.density_based import seqscan_labels

import nomad.visit_attribution.visit_attribution as visits
import nomad.filters as filters
import nomad.city_gen as cg

from nomad.map_utils import blocks_to_mercator_gdf
from nomad.contact_estimation import compute_stop_detection_metrics
from nomad.stop_detection.viz import plot_stops_barcode, plot_time_barcode, plot_stops, plot_pings, _plot_base_geometry, plot_circles

In [ ]:
with open('config_2_stops.json', 'r', encoding='utf-8') as f:
    config = json.load(f)

In [ ]:
poi_table = gpd.read_parquet(config["buildings_file"]).rename(columns={"id":"location"})
sparse_df = loader.from_file(config["output_files"]["sparse_path"], format="parquet")
diaries_df = loader.from_file(config["output_files"]["diaries_path"], format="parquet").rename(
    columns={"identifier":"user_id"})

In [ ]:
sparse_df = sparse_df[sparse_df['user_id'] == 'serene_yalow']
sparse_df.drop(columns=['datetime'], inplace=True)
diaries_df = diaries_df[diaries_df['user_id'] == 'serene_yalow']

In [ ]:
# Parameters
output = seqscan_labels(
    sparse_df,
    dist_thresh=3,
    time_thres=60,
    min_pts=2,
    back_merge=False,
    return_cores=True,
    traj_cols={'x': 'x', 'y': 'y', 'timestamp': 'timestamp'}
)
sparse_df['cluster'] = output.cluster
#sparse_df['cluster'] = labels

In [ ]:
if 'location' in sparse_df:
    sparse_df.drop(columns=["location"], inplace=True)
    
sparse_df["location"] = visits.point_in_polygon(
    data=sparse_df, 
    poi_table=poi_table.loc[poi_table.location.isin(['w-x17-y10', 'r-x19-y11'])], 
    data_crs='EPSG:3857',
    max_distance=20, 
    location_id='location', 
    method='centroid',
    x='x', 
    y='y'
)

In [ ]:
summarize_stops_with_loc = partial(
    utils.summarize_stop,
    x='x',
    y='y',
    keep_col_names=False,
    passthrough_cols=['location'],
    complete_output=False)

stops = sparse_df[sparse_df.cluster!=-1].groupby('cluster', as_index=False).apply(
            summarize_stops_with_loc, include_groups=False)
stops

In [ ]:
diaries_df

In [ ]:
metrics = compute_stop_detection_metrics(
            stops=stops,
            truth=diaries_df,
            user_id='serene_yalow',
            algorithm='seqscan',
            traj_cols={'location_id': 'location'},
            timestamp='timestamp'
        )
metrics

In [ ]:
backup = sparse_df

In [ ]:
# sparse_df = backup.query("1717217428 > timestamp > 1717214412")
# sparse_df = backup

In [ ]:
fig, (ax_map, ax_barcode) = plt.subplots(2, 1, figsize=(6,4.5),
                                         gridspec_kw={'height_ratios':[10,1]})
# Plot colored pings
ax_map.set_title("dist_tresh = 3")
plot_circles(sparse_df, ax=ax_map, radius=1.5, color='cluster', cmap='PiYG', base_geometry=poi_table, base_geom_color='#8c8c8c', base_geom_background='#d3d3d3')
plot_pings(sparse_df, ax=ax_map, s=2, color='black')
plot_time_barcode(sparse_df['timestamp'], ax=ax_barcode, set_xlim=True)
plot_time_barcode(sparse_df, color='cluster', ax=ax_barcode, cmap='PiYG', set_xlim=False, lw=1.3)

#plot_stops_barcode(diaries_df.dropna(), ax=ax_barcode, stop_alpha=0.6, cmap='inferno_r', set_xlim=False, timestamp='timestamp')

ax_barcode.set_title("timestamps")

plt.plot()